# TCC — Aplicação de modelos de machine learning na previsão do CDI
## Seção 4.3 — Avaliação preditiva: ML vs. DI implícita vs. CDI realizado

**Autor:** Marcos Paulo Galli dos Santos
**Orientador:** Prof. José Agostinho Baitello
**Centro Universitário FEI** — Bacharelado em Engenharia de Produção (2026)

Este notebook implementa os quatro subtópicos da seção 4.3:

- **4.3.1** — Métricas agregadas no período de teste (MAE, RMSE, MAPE)
- **4.3.2** — Teste de Diebold-Mariano (perda quadrática, HAC Bartlett 21 lags,
  correção HLN)
- **4.3.3** — Análise de trajetória ao longo do teste (Figuras 9 e 10,
  viés sistemático, sub-períodos delimitados pelas reuniões do Copom)
- **4.3.4** — Pontos de inflexão e momentos de divergência entre ML e DI

A 4.3 não retrina o modelo — carrega o `modelo_xgb_final.joblib` produzido pela 4.2
e gera as previsões diretamente sobre o conjunto de teste (02/01 a 31/03/2026, 61 obs).

Datas do Copom no período de teste, conforme calendário oficial em
`Calendario_Copom`: **28/01/2026 e 18/03/2026** (ambas quartas-feiras).

In [ ]:
## 0. Setup do ambiente

# Instala XGBoost se ainda não estiver disponível (Colab já vem com pandas, sklearn, etc.)
import subprocess, sys
print("xgboost", xgboost.__version__)

# Upload dos arquivos no Colab.
# Faça upload de TRÊS arquivos:
#   - dados_modelo_tcc_cdi.xlsx
#   - dados/dados_modelo_tcc_di_futuro.xlsx
#   - modelo_xgb_final.joblib  (gerado pelo notebook da 4.2)

import os, time, warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from scipy import stats as scistats
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
import joblib

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Parâmetros do experimento (espelham 4.2)
PATH_CDI   = "dados/dados_modelo_tcc_cdi.xlsx"
PATH_DI = "dados/dados_modelo_tcc_di_futuro.xlsx"
PATH_MODEL = "modelo_xgb_final.joblib"

DATA_TREINO_INI  = pd.Timestamp("2023-01-02")
DATA_TREINO_FIM  = pd.Timestamp("2025-11-28")
DATA_EMBARGO_INI = pd.Timestamp("2025-12-01")
DATA_EMBARGO_FIM = pd.Timestamp("2026-01-01")
DATA_TESTE_INI   = pd.Timestamp("2026-01-02")
DATA_TESTE_FIM   = pd.Timestamp("2026-03-31")

H = 22  # horizonte de previsão em dias úteis

# Paleta de cores consistente entre figuras
COR_REAL = "#000000"   # CDI realizado — referência
COR_ML   = "#1f4e79"   # XGBoost
COR_DI   = "#c0504d"   # Curva DI implícita
COR_NAIVE = "#7f7f7f"  # Benchmark trivial


## 4.3.0 — Carregamento dos artefatos da 4.2 e construção das três séries

Carrega: (i) o modelo XGBoost final salvo em `.joblib` pela 4.2; (ii) a base
consolidada já tratada conforme 3.5, com o CDI realizado e todas as 73 features;
(iii) as cotações diárias da curva DI Futuro coletadas via tvDatafeed.

Reconstrói o alvo `y_22d` sobre a base completa e isola o período de teste.
Gera as três séries que serão comparadas:

- **Previsão XGBoost**: `model.predict(X_teste)` — usando o modelo carregado
- **Taxa implícita DI**: para cada data t, o contrato cujo vencimento está mais
  próximo de t+22du (B3 padrão: 1º dia útil do mês do vencimento)
- **Benchmark "sem mudança"**: `CDI(t)` repetido — previsão trivial

In [ ]:
# Modelo final salvo pela 4.2
artefatos = joblib.load(PATH_MODEL)
model         = artefatos["model"]
feature_cols  = artefatos["feature_cols"]
best_params   = artefatos["best_params"]
rmse_cv_medio = artefatos.get("rmse_cv_medio", None)

print(f"Modelo carregado: {type(model).__name__}")
print(f"Número de features esperadas: {len(feature_cols)}")
print(f"Hiperparâmetros: {best_params}")
print(f"RMSE médio na CV (4.2): {rmse_cv_medio:.4f} p.p." if rmse_cv_medio else "")

# Base consolidada e reconstrução do alvo (idêntico à 4.2)
df = pd.read_excel(PATH_CDI, sheet_name="Diario_Consolidado")
df["Data"] = pd.to_datetime(df["Data"])
df = df.sort_values("Data").reset_index(drop=True)

df["y_22d"] = (
    df["CDI"].shift(-1).rolling(window=H, min_periods=H).mean().shift(-(H - 1))
)
print(f"Base consolidada: {df.shape[0]} obs ({df['Data'].min().date()} a {df['Data'].max().date()})")

# Calendário Copom — usado para QA e para sombrear figuras
copom = pd.read_excel(PATH_CDI, sheet_name="Calendario_Copom")
copom["Reunioes_Copom"] = pd.to_datetime(copom["Reunioes_Copom"])
copom_teste = copom[
    (copom["Reunioes_Copom"] >= DATA_TESTE_INI) &
    (copom["Reunioes_Copom"] <= DATA_TESTE_FIM)
].copy().reset_index(drop=True)
print(f"Reuniões Copom no teste: {copom_teste['Reunioes_Copom'].dt.date.tolist()}")

# Recorte do teste — features, y_real e auditoria de integridade
mask_teste = (df["Data"] >= DATA_TESTE_INI) & (df["Data"] <= DATA_TESTE_FIM)
df_teste = df.loc[mask_teste].reset_index(drop=True)
X_teste      = df_teste[feature_cols].copy()
y_real_teste = df_teste["y_22d"].copy()
dates_teste  = df_teste["Data"].copy()

print(f"\nPeríodo de teste: {dates_teste.min().date()} a {dates_teste.max().date()}")
print(f"Observações: {len(df_teste)}")
print(f"NaN no alvo (deve ser zero): {y_real_teste.isna().sum()}")

# Auditoria de NaN nas features do teste — relevante porque variáveis Focus_*_anoproximo_*
# podem estar ausentes dependendo do horizonte coletado da API Olinda
nan_teste = X_teste.isna().sum()
nan_teste = nan_teste[nan_teste > 0].sort_values(ascending=False)
print(f"\nFeatures com NaN no teste: {len(nan_teste)} de {len(feature_cols)}")
if len(nan_teste):
    print(nan_teste.to_string())
    print("XGBoost lida com NaN nativamente (default direction nos splits).")


### 4.3.0.a — Curva DI Futuro e inferência dos vencimentos

Os quatro contratos coletados (G/H/J/K 2026) vencem no primeiro dia útil do mês
do código, conforme padrão B3 para DI1. A função abaixo decodifica o código e
ajusta para o próximo dia útil quando o dia 1 cai em fim de semana ou feriado
nacional (1º de maio, por exemplo).

In [ ]:
di = pd.read_excel(PATH_DI, sheet_name="DI_Futuro")
di["Data"] = pd.to_datetime(di["Data"])

# Conversão: Taxa_a.a. vem em decimal (0.14895), padronizar em % a.a. (14.895)
di["Taxa_perc"] = di["Taxa_a.a."] * 100.0

# Decodificação do código do contrato (B3 padrão): DI1{LETRA}{ANO}
# F=Jan, G=Feb, H=Mar, J=Apr, K=May, M=Jun, N=Jul, Q=Aug, U=Sep, V=Oct, X=Nov, Z=Dec
LETRA_MES = {"F":1,"G":2,"H":3,"J":4,"K":5,"M":6,
             "N":7,"Q":8,"U":9,"V":10,"X":11,"Z":12}

# Feriados nacionais com impacto em 2026 (B3) — para ajustar vencimentos no 1º dia útil
FERIADOS_2026 = {pd.Timestamp("2026-01-01"), pd.Timestamp("2026-02-16"),
                 pd.Timestamp("2026-02-17"), pd.Timestamp("2026-02-18"),
                 pd.Timestamp("2026-04-03"), pd.Timestamp("2026-04-21"),
                 pd.Timestamp("2026-05-01"), pd.Timestamp("2026-06-04"),
                 pd.Timestamp("2026-09-07"), pd.Timestamp("2026-10-12"),
                 pd.Timestamp("2026-11-02"), pd.Timestamp("2026-11-15"),
                 pd.Timestamp("2026-11-20"), pd.Timestamp("2026-12-25")}

def proximo_dia_util(d):
    while (d.weekday() >= 5) or (d in FERIADOS_2026):
        d = d + pd.Timedelta(days=1)
    return d

def codigo_para_vencimento(codigo):
    letra = codigo[3]
    ano   = int(codigo[4:8])
    mes   = LETRA_MES[letra]
    return proximo_dia_util(pd.Timestamp(year=ano, month=mes, day=1))

di["Vencimento"] = di["Contrato"].apply(codigo_para_vencimento)

# Resumo: cada contrato e seu vencimento
resumo_contratos = (
    di.groupby("Contrato")
      .agg(Vencimento=("Vencimento", "first"),
           Data_inicio=("Data", "min"),
           Data_fim=("Data", "max"),
           N_pregoes=("Data", "count"))
      .reset_index()
      .sort_values("Vencimento")
)
print("Contratos e vencimentos inferidos:")
print(resumo_contratos.to_string(index=False))


### 4.3.0.b — Taxa implícita do contrato com vencimento mais próximo a t+22du

Para cada data t do teste, identifica-se a data alvo t+22du (calendário efetivo
da B3, dado pela própria sequência de datas em `Diario_Consolidado`). Entre os
contratos cotados em t, seleciona-se aquele cujo vencimento minimiza
|Vencimento − (t+22du)| em dias úteis.

In [ ]:
# Calendário de dias úteis efetivos da B3 (extraído da sequência de Data no Diario_Consolidado)
calendario_du = df["Data"].sort_values().reset_index(drop=True)
mapa_du = {d: i for i, d in enumerate(calendario_du)}

def shift_22_du(t):
    """Retorna a data correspondente a t + H dias úteis."""
    if t not in mapa_du:
        return None
    idx = mapa_du[t]
    return calendario_du.iloc[idx + H] if idx + H < len(calendario_du) else None

def du_entre(d1, d2):
    """Dias úteis entre d1 e d2 (positivo se d2 > d1) — usa o calendário B3."""
    # Pega o índice mais próximo no calendário B3 para datas que possam ser fim de semana/feriado
    serie = calendario_du
    i1 = serie.searchsorted(d1)
    i2 = serie.searchsorted(d2)
    return int(i2 - i1)

implicit_di_rows = []
for t in dates_teste:
    t = pd.Timestamp(t)
    alvo = shift_22_du(t)
    di_em_t = di[di["Data"] == t].copy()
    if len(di_em_t) == 0 or alvo is None:
        implicit_di_rows.append({"Data": t, "Contrato": None, "Vencimento": None,
                                 "DU_ate_venc": None, "DU_diff_22": None,
                                 "Taxa_implicita_22du_pct": np.nan})
        continue
    di_em_t["DU_ate_venc"] = di_em_t["Vencimento"].apply(lambda v: du_entre(t, v))
    di_em_t["DU_diff_22"] = (di_em_t["DU_ate_venc"] - H).abs()
    melhor = di_em_t.sort_values("DU_diff_22").iloc[0]
    implicit_di_rows.append({
        "Data":        t,
        "Contrato":    melhor["Contrato"],
        "Vencimento":  melhor["Vencimento"],
        "DU_ate_venc": int(melhor["DU_ate_venc"]),
        "DU_diff_22":  int(melhor["DU_diff_22"]),
        "Taxa_implicita_22du_pct": float(melhor["Taxa_perc"]),
    })

implicit_di_df = pd.DataFrame(implicit_di_rows)
print("Contrato usado para cada data do teste (5 primeiras e 5 últimas):")
print(pd.concat([implicit_di_df.head(5), implicit_di_df.tail(5)]).to_string(index=False))

# Distribuição: quantos dias usaram cada contrato?
print("\nDistribuição de contratos usados como referência:")
print(implicit_di_df["Contrato"].value_counts().to_string())

# Distribuição do gap entre DU real até o vencimento e os 22 du teóricos
print(f"\nGap DU_diff_22 — mediana: {implicit_di_df['DU_diff_22'].median():.0f} | "
      f"máximo: {implicit_di_df['DU_diff_22'].max():.0f}")


### 4.3.0.c — Geração das previsões e consolidação

Empacota o CDI realizado, as três previsões e o contrato DI escolhido em cada
data num único DataFrame `resultados`, que será usado em todas as subseções
seguintes.

In [ ]:
# Previsão XGBoost
y_pred_ml = pd.Series(model.predict(X_teste), index=dates_teste, name="Previsao_ML")

# Benchmark trivial "sem mudança": CDI(t) replicado
y_pred_naive = df_teste.set_index("Data")["CDI"].copy()
y_pred_naive.index = dates_teste
y_pred_naive.name = "Previsao_Naive"

# Taxa implícita DI alinhada com dates_teste
y_pred_di = pd.Series(
    implicit_di_df.set_index("Data")["Taxa_implicita_22du_pct"].reindex(dates_teste).values,
    index=dates_teste, name="DI_Implicita"
)

resultados = pd.DataFrame({
    "Data":          dates_teste.values,
    "CDI_realizado": y_real_teste.values,
    "Previsao_ML":   y_pred_ml.values,
    "DI_Implicita":  y_pred_di.values,
    "Previsao_Naive": y_pred_naive.values,
    "Contrato_DI":   implicit_di_df["Contrato"].values,
    "Venc_DI":       implicit_di_df["Vencimento"].values,
    "DU_diff_22":    implicit_di_df["DU_diff_22"].values,
})

# Erros (verdade − previsão) — sinal preserva o "viés"
resultados["Erro_ML"]    = resultados["CDI_realizado"] - resultados["Previsao_ML"]
resultados["Erro_DI"]    = resultados["CDI_realizado"] - resultados["DI_Implicita"]
resultados["Erro_Naive"] = resultados["CDI_realizado"] - resultados["Previsao_Naive"]

print("Resultados — head:")
print(resultados.head(5).to_string(index=False))
print("\nResultados — tail:")
print(resultados.tail(5).to_string(index=False))

resultados.to_excel("resultados_43_consolidado.xlsx", index=False)

# Identificação das datas sem y_22d completo:
# O calendário B3 de 2026 tem feriados de Páscoa (03/04, Sexta Santa) e Tiradentes
# (21/04) que reduzem abril a 20 du efetivos. Como a base do CDI termina em
# 30/04/2026 e o alvo exige 22 du de realização, as duas últimas previsões do
# teste (30/03 e 31/03) ficam sem y_22d completo: o 22º du futuro cairia em
# ~04/05 e ~05/05, fora da base.
mask_validos = resultados["CDI_realizado"].notna() & resultados["DI_Implicita"].notna()
n_total    = len(resultados)
n_validos  = int(mask_validos.sum())
n_drop     = n_total - n_validos
datas_drop = resultados.loc[~mask_validos, "Data"].dt.date.tolist()

print(f"\nObservações do teste com horizonte realizado completo: {n_validos} de {n_total}")
if n_drop > 0:
    print(f"Datas excluídas das métricas por falta de y_22d completo: {datas_drop}")
    print("(Previsões dessas datas permanecem em 'resultados' para inspeção,")
    print(" mas Tabelas 7–10 são calculadas apenas sobre as observações válidas.)")

resultados_validos = resultados.loc[mask_validos].reset_index(drop=True)
print(f"\nPeríodo de avaliação efetivo: "
      f"{resultados_validos['Data'].min().date()} a "
      f"{resultados_validos['Data'].max().date()} "
      f"(n={n_validos})")


## 4.3.1 — Métricas agregadas no período de teste

Tabela 7 — núcleo quantitativo da seção. Compara MAE, RMSE e MAPE de cada
abordagem contra o CDI efetivamente realizado nos 22 du seguintes a cada data t
do teste. O conjunto efetivo de avaliação tem **59 observações** (02/01 a
27/03/2026), excluindo 30/03 e 31/03 por ausência de horizonte realizado completo.

O benchmark trivial "sem mudança" é incluído deliberadamente: nos sub-períodos
de estabilidade do CDI ele é estruturalmente competitivo, o que torna sua
inclusão útil como piso de comparabilidade. O Copom de 18/03/2026 promoveu
corte de 25 bps (Selic 15,00 → 14,75, CDI 14,90 → 14,65 efetivo em 19/03),
o que dá amplitude relevante para a comparação.

In [ ]:
def mae(y, yhat):
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def mape(y, yhat):
    return float(np.mean(np.abs((y - yhat) / y)) * 100.0)

y_obs = resultados_validos["CDI_realizado"].values

tabela_7 = pd.DataFrame({
    "Modelo XGBoost": [
        mae (y_obs, resultados_validos["Previsao_ML"].values),
        rmse(y_obs, resultados_validos["Previsao_ML"].values),
        mape(y_obs, resultados_validos["Previsao_ML"].values),
    ],
    "Curva DI implícita (22du)": [
        mae (y_obs, resultados_validos["DI_Implicita"].values),
        rmse(y_obs, resultados_validos["DI_Implicita"].values),
        mape(y_obs, resultados_validos["DI_Implicita"].values),
    ],
    "Benchmark trivial (sem mudança)": [
        mae (y_obs, resultados_validos["Previsao_Naive"].values),
        rmse(y_obs, resultados_validos["Previsao_Naive"].values),
        mape(y_obs, resultados_validos["Previsao_Naive"].values),
    ],
}, index=["MAE (p.p.)", "RMSE (p.p.)", "MAPE (%)"]).T

print(f"Tabela 7 — Métricas agregadas no período de teste "
      f"({resultados_validos['Data'].min().date()} a "
      f"{resultados_validos['Data'].max().date()}, n={n_validos})")
print(tabela_7.round(4).to_string())
tabela_7.to_excel("tabela_7_metricas_agregadas.xlsx")

# Para contextualizar magnitudes: amplitude do CDI realizado no período de avaliação
amplitude = pd.Series({
    "Mín CDI realizado":    resultados_validos["CDI_realizado"].min(),
    "Máx CDI realizado":    resultados_validos["CDI_realizado"].max(),
    "Amplitude (p.p.)":     resultados_validos["CDI_realizado"].max() - resultados_validos["CDI_realizado"].min(),
    "Desvio-padrão (p.p.)": resultados_validos["CDI_realizado"].std(ddof=0),
})
print("\nContexto do CDI realizado no período de avaliação:")
print(amplitude.round(4).to_string())


## 4.3.2 — Teste de Diebold-Mariano

Especificação operacional adotada:

- **Função de perda**: quadrática, `L(e) = e^2`
- **Variância de longa duração**: estimador HAC de Newey-West com kernel
  Bartlett e bandwidth `h-1 = 21` (h=22 dias úteis é o horizonte do alvo).
  Garante variância não-negativa em qualquer amostra.
- **Correção de pequena amostra**: Harvey-Leybourne-Newbold (1997), que ajusta
  a estatística DM e usa distribuição t de Student com `n-1` graus de
  liberdade. Apropriado para n=59.
- **Comparações pareadas**: três pares — ML vs. DI, ML vs. Naive, DI vs. Naive.

**Observação metodológica:** dado o tamanho amostral curto (n=59) e a
amplitude limitada do CDI realizado no período de avaliação (~0,26 p.p.),
p-valores grandes (não-significância) são esperados e devem ser interpretados
como informação sobre o ambiente — e não como falha do modelo.

In [ ]:
def hac_variance_bartlett(d, lag):
    """Variância de longa duração com kernel Bartlett (Newey-West)."""
    n = len(d)
    d_bar = float(np.mean(d))
    gamma0 = float(np.mean((d - d_bar) ** 2))
    soma = 0.0
    for k in range(1, lag + 1):
        w = 1.0 - k / (lag + 1.0)  # kernel Bartlett
        cov = float(np.mean((d[k:] - d_bar) * (d[:-k] - d_bar)))
        soma += 2.0 * w * cov
    return max(gamma0 + soma, 1e-12)

def diebold_mariano(e1, e2, h=22, loss="quadratic"):
    """Diebold-Mariano com HAC Bartlett (h-1 lags) e correção HLN.

    Retorna dict com DM original, DM corrigido HLN, p-valor bicaudal,
    diferencial médio de perda e n.
    """
    e1 = np.asarray(e1, dtype=float)
    e2 = np.asarray(e2, dtype=float)
    if loss == "quadratic":
        d = e1 ** 2 - e2 ** 2
    elif loss == "absolute":
        d = np.abs(e1) - np.abs(e2)
    else:
        raise ValueError(loss)
    n = len(d)
    d_bar = float(np.mean(d))
    sigma2_LR = hac_variance_bartlett(d, lag=h - 1)
    dm = d_bar / np.sqrt(sigma2_LR / n)
    correction = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm_hln = dm * correction
    p = 2.0 * (1.0 - scistats.t.cdf(abs(dm_hln), df=n - 1))
    return {
        "n": n,
        "d_bar": d_bar,
        "DM": float(dm),
        "DM_HLN": float(dm_hln),
        "p_valor": float(p),
    }

pares = [
    ("XGBoost",      "DI implícita", resultados_validos["Erro_ML"].values, resultados_validos["Erro_DI"].values),
    ("XGBoost",      "Naive",        resultados_validos["Erro_ML"].values, resultados_validos["Erro_Naive"].values),
    ("DI implícita", "Naive",        resultados_validos["Erro_DI"].values, resultados_validos["Erro_Naive"].values),
]

linhas_dm = []
for a, b, ea, eb in pares:
    r = diebold_mariano(ea, eb, h=H, loss="quadratic")
    linhas_dm.append({
        "Comparação (modelo A vs modelo B)": f"{a} vs {b}",
        "n":               r["n"],
        "Diferencial médio d̄":  r["d_bar"],
        "DM":              r["DM"],
        "DM (HLN)":        r["DM_HLN"],
        "p-valor (bicaudal)": r["p_valor"],
    })

tabela_8 = pd.DataFrame(linhas_dm)
print("Tabela 8 — Teste de Diebold-Mariano (perda quadrática, HAC Bartlett 21 lags, correção HLN)")
print(tabela_8.round(4).to_string(index=False))
tabela_8.to_excel("tabela_8_diebold_mariano.xlsx", index=False)

print("\nInterpretação do sinal de d̄ (perda quadrática):")
print(" - d̄ < 0  ==> modelo A tem MSE menor (mais acurado) que B")
print(" - d̄ > 0  ==> modelo A tem MSE maior (menos acurado) que B")


## 4.3.3 — Análise de trajetória ao longo do teste

Três produtos:

1. **Figura 9** — três séries sobrepostas: CDI realizado, previsão XGBoost,
   taxa implícita DI. Linhas verticais marcam as duas reuniões do Copom no
   período (28/01 e 18/03/2026, confirmadas no calendário da base). Como a
   reunião de março promoveu corte de 25 bps, a Figura mostra três regimes
   visualmente distintos: estabilidade em janeiro, queda gradual da `y_22d`
   média ao longo de fevereiro/início de março, e estabilidade no novo
   patamar a partir de 19/03.
2. **Figura 10** — erros diários (CDI realizado − previsão) para ML e DI,
   com as mesmas linhas verticais. Permite ler sinal do viés e amplitude de
   cada projeção. As duas últimas datas (30/03 e 31/03) aparecem sem ponto
   por falta de `y_22d`.
3. **Tabela 9** — viés sistemático e MAE por três sub-períodos: pré-Copom de
   janeiro, entre as duas reuniões, e pós-Copom de março. O sub-período
   pós-março corre apenas até 27/03 (oito dias úteis com horizonte completo).

In [ ]:
# Figura 9 — três séries sobrepostas
fig9, ax = plt.subplots(figsize=(11, 5.2))
ax.plot(resultados["Data"], resultados["CDI_realizado"],
        label="CDI realizado", color=COR_REAL, linewidth=2.0, zorder=3)
ax.plot(resultados["Data"], resultados["Previsao_ML"],
        label="Previsão XGBoost", color=COR_ML, linewidth=1.6, zorder=2)
ax.plot(resultados["Data"], resultados["DI_Implicita"],
        label="Taxa implícita DI (22 du)", color=COR_DI, linewidth=1.6, zorder=2)

for d_copom in copom_teste["Reunioes_Copom"]:
    ax.axvline(d_copom, color="gray", alpha=0.4, linestyle="--", linewidth=1.0, zorder=1)
    ax.annotate(f"Copom {d_copom.strftime('%d/%m')}",
                xy=(d_copom, ax.get_ylim()[1] if False else resultados["CDI_realizado"].max()),
                xytext=(4, -12), textcoords="offset points",
                fontsize=8, color="gray")

ax.set_xlabel("Data")
ax.set_ylabel("Taxa (% a.a.)")
ax.set_title("Figura 9 — CDI realizado, previsão XGBoost e taxa implícita DI "
             "(jan–mar/2026)")
ax.legend(loc="lower left", framealpha=0.9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("figura_9_trajetoria.png", dpi=180, bbox_inches="tight")
plt.show()

# Figura 10 — erros diários
fig10, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(resultados["Data"], resultados["Erro_ML"],
        label="Erro XGBoost", color=COR_ML, linewidth=1.4)
ax.plot(resultados["Data"], resultados["Erro_DI"],
        label="Erro DI implícita", color=COR_DI, linewidth=1.4)
ax.axhline(0, color="black", linewidth=0.6)

for d_copom in copom_teste["Reunioes_Copom"]:
    ax.axvline(d_copom, color="gray", alpha=0.4, linestyle="--", linewidth=1.0)

ax.set_xlabel("Data")
ax.set_ylabel("Erro (p.p.) = CDI realizado − previsão")
ax.set_title("Figura 10 — Erros diários: XGBoost e DI implícita")
ax.legend(loc="best", framealpha=0.9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("figura_10_erros_diarios.png", dpi=180, bbox_inches="tight")
plt.show()

# Viés sistemático (erro médio com sinal) — somente para evitar ambiguidade,
# defino viés como E[CDI − previsão]: > 0 ==> previsão SUBESTIMOU o realizado
bias = pd.Series({
    "Modelo XGBoost":                 resultados_validos["Erro_ML"].mean(),
    "Curva DI implícita (22du)":      resultados_validos["Erro_DI"].mean(),
    "Benchmark trivial (sem mudança)": resultados_validos["Erro_Naive"].mean(),
}, name="Viés (p.p.)")
print("Viés sistemático (CDI realizado − previsão):")
print(bias.round(4).to_string())

# Tabela 9 — sub-períodos delimitados pelo Copom
copom1 = pd.Timestamp(copom_teste["Reunioes_Copom"].iloc[0])  # 28/01/2026
copom2 = pd.Timestamp(copom_teste["Reunioes_Copom"].iloc[1])  # 18/03/2026

sub_def = [
    ("Pré-Copom de janeiro (02/01 até véspera de 28/01)",
        resultados_validos["Data"] < copom1),
    ("Entre as duas reuniões (28/01 inclusivo até véspera de 18/03)",
        (resultados_validos["Data"] >= copom1) & (resultados_validos["Data"] < copom2)),
    ("Pós-Copom de março (18/03 até 27/03)",
        resultados_validos["Data"] >= copom2),
]

sub_rows = []
for nome, mask in sub_def:
    if mask.sum() == 0:
        continue
    r = resultados_validos[mask]
    yt = r["CDI_realizado"].values
    sub_rows.append({
        "Sub-período":   nome,
        "N obs":         int(mask.sum()),
        "MAE ML":        mae(yt, r["Previsao_ML"].values),
        "MAE DI":        mae(yt, r["DI_Implicita"].values),
        "MAE Naive":     mae(yt, r["Previsao_Naive"].values),
        "Viés ML":       float((yt - r["Previsao_ML"].values).mean()),
        "Viés DI":       float((yt - r["DI_Implicita"].values).mean()),
        "Viés Naive":    float((yt - r["Previsao_Naive"].values).mean()),
    })
tabela_9 = pd.DataFrame(sub_rows)
print("\nTabela 9 — Métricas por sub-período")
print(tabela_9.round(4).to_string(index=False))
tabela_9.to_excel("tabela_9_subperiodos.xlsx", index=False)


## 4.3.4 — Pontos de inflexão e momentos de divergência

Identificação dos dias do teste em que |XGBoost − DI implícita| foi maior, com
caracterização macro: presença de reunião do Copom (e janelas pré/pós),
dummies de cauda do Ibovespa, surpresa de IPCA (`Surpresa_IPCA`) e velocidade
de revisão do Focus.

O foco aqui é qualitativo: dado o tamanho amostral, mais do que estabelecer
testes formais, interessa observar se a divergência se concentra em
contextos com sinais informacionais identificáveis ou se é difusa.

In [ ]:
# Métrica de divergência (sinal preservado: positivo ==> ML > DI)
# Calcula sobre resultados_validos para garantir que toda divergência reportada
# corresponde a uma data com CDI realizado completo e contrato DI definido.
resultados_validos = resultados_validos.copy()
resultados_validos["Divergencia_ML_DI"]     = resultados_validos["Previsao_ML"] - resultados_validos["DI_Implicita"]
resultados_validos["Divergencia_ML_DI_abs"] = resultados_validos["Divergencia_ML_DI"].abs()

print("Estatísticas da divergência ML − DI no período de avaliação (n=59):")
print(resultados_validos["Divergencia_ML_DI"].describe().round(4).to_string())

# Top-10 dias de maior divergência (em módulo)
top_div = resultados_validos.nlargest(10, "Divergencia_ML_DI_abs")[[
    "Data", "CDI_realizado", "Previsao_ML", "DI_Implicita",
    "Divergencia_ML_DI", "Contrato_DI",
]].reset_index(drop=True)

# Cruzamento com features de contexto macro/comportamental
contexto_cols = [
    "Data",
    "Copom_Dia", "Copom_PreReuniao", "Copom_Semana",
    "Copom_Surpresa_Direcao_5du", "Copom_Surpresa_Magnitude_5du",
    "Copom_Surpresa_Direcao_22du", "Copom_Surpresa_Magnitude_22du",
    "Ibov_Cauda_Inferior", "Ibov_Cauda_Superior",
    "Surpresa_IPCA",
    "Revisao_Focus_Selic", "Revisao_Focus_IPCA",
]
contexto = df[contexto_cols].copy()
tabela_10 = top_div.merge(contexto, on="Data", how="left")

print("\nTabela 10 — Top-10 dias com maior |XGBoost − DI implícita| e contexto macro")
print(tabela_10.round(4).to_string(index=False))
tabela_10.to_excel("tabela_10_pontos_divergencia.xlsx", index=False)

# Frequência relativa: em quantos desses 10 dias houve sinal de Copom/cauda Ibov/IPCA
sinais = {
    "Dias em janela Copom (PreReuniao=1 ou Semana=1)":
        int(((tabela_10["Copom_PreReuniao"] == 1) | (tabela_10["Copom_Semana"] == 1)).sum()),
    "Dias com Surpresa_Magnitude_5du > 0":
        int((tabela_10["Copom_Surpresa_Magnitude_5du"].abs() > 0).sum()),
    "Dias com Ibov em cauda (sup ou inf)":
        int(((tabela_10["Ibov_Cauda_Inferior"] == 1) | (tabela_10["Ibov_Cauda_Superior"] == 1)).sum()),
    "Dias com |Surpresa_IPCA| > 0 (divulgação de IPCA)":
        int((tabela_10["Surpresa_IPCA"].abs() > 0).sum()),
    "Dias com |Revisao_Focus_Selic| > 0 (publicação Focus)":
        int((tabela_10["Revisao_Focus_Selic"].abs() > 0).sum()),
}
print("\nResumo qualitativo: contexto macro nos 10 dias de maior divergência")
for k, v in sinais.items():
    print(f"  {k:55s}: {v}/10")


## Encerramento — persistência de artefatos e download

Salva todas as tabelas, figuras e o DataFrame consolidado da 4.3 para uso na
redação do capítulo.

In [ ]:
artefatos_43 = [
    "resultados_43_consolidado.xlsx",
    "tabela_7_metricas_agregadas.xlsx",
    "tabela_8_diebold_mariano.xlsx",
    "tabela_9_subperiodos.xlsx",
    "tabela_10_pontos_divergencia.xlsx",
    "figura_9_trajetoria.png",
    "figura_10_erros_diarios.png",
]
print("Artefatos gerados:")
for nome in artefatos_43:
    if os.path.exists(nome):
        print(f"  ✔ {nome}  ({os.path.getsize(nome)/1024:.1f} KB)")
    else:
        print(f"  ✘ {nome}  (não encontrado)")

# Download (Colab)
for nome in artefatos_43:
    if os.path.exists(nome):
        files.download(nome)
